In [1]:
## Install Packages
import pandas as pd
import requests
import io

In [2]:
# 福岡市　登録人口（行政区別・人口動態）
# https://data.bodik.jp/dataset/401307_population_touroku_demographic
import requests
import pandas as pd
import re

def fetch_fukuoka_data_api(resource_id, limit=1000):
    # CKANのdatastore_searchエンドポイント
    api_url = 'https://data.bodik.jp/api/3/action/datastore_search'

    # パラメータの設定
    params = {
        'resource_id': resource_id,
        'limit': limit,
        # 'q': '中央区'  # キーワード検索が必要な場合はここを追加
    }

    response = requests.get(api_url, params=params)

    if response.status_code == 200:
        data = response.json()
        # 結果のレコード部分をDataFrameに変換
        records = data['result']['records']
        return pd.DataFrame(records)
    else:
        print(f"APIエラー: {response.status_code}")
        return None

# 複数のリソースIDを指定（以前のURLリストから抽出）
# 注：このリストは以前の会話履歴に基づいています。
resource_ids = [
    "804da11b-7b58-4702-8ce4-72b793e4c1c4", # 令和7年12月末日現在
    "6ddb99d6-defc-4762-9a4e-db3553ad450b", # 令和7年11月末日現在
    "515daeef-7e59-4b1a-8f02-e3b559410aba", # 令和7年10月末日現在
    "3b68d357-db40-42bb-9ff6-380871b13705", # 令和7年9月末日現在
    "1b37a848-2a54-4c9f-8799-75d5a75ba995", # 令和7年8月末日現在
    "8aa5244f-6d8f-43df-9d1f-431f820118f5", # 令和7年7月末日現在
    "fce50d84-9880-4c70-91f4-1de0aea2bede", # 令和7年6月末日現在
    "d6108240-a6aa-47fd-86a5-2ffe9dd9361a", # 令和7年5月末日現在
    "e993fe0e-58d6-4e14-a4b7-79fe78c16a05", # 令和7年4月末日現在
    "9eaa9562-e830-4b5a-ab70-c755df8e793e", # 令和7年3月末日現在
    "518464fe-ddb0-48f8-b953-873e6ddf773a", # 令和7年2月末日現在
    "cb4108d2-5018-438b-9226-648bb850652f", # 令和7年1月末日現在
    "f87dee12-c66b-4b2a-917e-aca35dc8f86c", # 令和6年12月末日現在
    "0419a647-55f3-41a5-b8d4-ee3c50553aca", # 令和6年11月末日現在
    "65ba896b-e3f3-4373-ae31-0cb9d462de9f", # 令和6年10月末日現在
    "4cc97bb7-bab6-4d43-86b6-d63ef08de0e5", # 令和6年9月末日現在
    "34c2f6dc-8c53-4bb7-b342-61855eef1967", # 令和6年8月末日現在
    "3d22050c-9b61-470f-aad0-c4f3f753bb74", # 令和6年7月末日現在
    "84c15c55-517a-4cc9-8298-41326adef084", # 令和6年6月末日現在
    "5cbb3337-7969-414e-bdc1-0af75edc6ca0", # 令和6年5月末日現在
    "86fcecae-9c1f-483e-a1b4-b22d97f5beb8", # 令和6年4月末日現在
    "e78f9ff3-5a45-4b6b-b919-0c2342c99e0d", # 令和6年3月末日現在
    "75527df5-53c1-424b-878a-e091c0e718a2", # 令和6年2月末日現在
    "8b81220d-6253-491d-a0d4-1bfacfa72672"  # 令和6年1月末日現在
]

all_dfs = []
for r_id in resource_ids:
    print(f"Fetching data for resource_id: {r_id}")
    df_temp = fetch_fukuoka_data_api(r_id)
    if df_temp is not None:
        all_dfs.append(df_temp)

# すべてのデータフレームを結合
if all_dfs:
    df_merged = pd.concat(all_dfs, ignore_index=True)
    print("すべてのデータフレームを結合しました。")

    # 1. 数値項目を確実に数値型へ
    num_cols = ['自然動態_増減', '社会動態_増減', '社会動態_市外からの転入', '社会動態_市外からの転出']
    # エラーが発生しないように、存在しないカラムは除外
    cols_to_convert = [col for col in num_cols if col in df_merged.columns]
    df_merged[cols_to_convert] = df_merged[cols_to_convert].apply(pd.to_numeric, errors='coerce')

    # 2. 日本人のデータのみを対象に各区を比較
    # '日本人_総数' を除外する
    df_jp = df_merged[df_merged['区分'].str.contains('日本人_', na=False) & (df_merged['区分'] != '日本人_総数')].copy()

    # 3. 「市外流入依存度」を計算（新しい住民による上書き率の指標）
    # ゼロ除算を避けるために分母が0でないことを確認
    denominator = df_jp['社会動態_市外からの転入'] + df_jp['社会動態_他区からの転入']
    df_jp['流入依存度'] = df_jp.apply(lambda row: row['社会動態_市外からの転入'] / denominator[row.name] if denominator[row.name] != 0 else 0, axis=1)

    # 結果の表示
    display(df_jp[['調査年月日', '区分', '流入依存度', '社会動態_増減']].sort_values(by=['調査年月日', '流入依存度'], ascending=[False, False]))
else:
    print("データを取得できませんでした。")

Fetching data for resource_id: 804da11b-7b58-4702-8ce4-72b793e4c1c4
Fetching data for resource_id: 6ddb99d6-defc-4762-9a4e-db3553ad450b
Fetching data for resource_id: 515daeef-7e59-4b1a-8f02-e3b559410aba
Fetching data for resource_id: 3b68d357-db40-42bb-9ff6-380871b13705
Fetching data for resource_id: 1b37a848-2a54-4c9f-8799-75d5a75ba995
Fetching data for resource_id: 8aa5244f-6d8f-43df-9d1f-431f820118f5
Fetching data for resource_id: fce50d84-9880-4c70-91f4-1de0aea2bede
Fetching data for resource_id: d6108240-a6aa-47fd-86a5-2ffe9dd9361a
Fetching data for resource_id: e993fe0e-58d6-4e14-a4b7-79fe78c16a05
Fetching data for resource_id: 9eaa9562-e830-4b5a-ab70-c755df8e793e
Fetching data for resource_id: 518464fe-ddb0-48f8-b953-873e6ddf773a
Fetching data for resource_id: cb4108d2-5018-438b-9226-648bb850652f
Fetching data for resource_id: f87dee12-c66b-4b2a-917e-aca35dc8f86c
Fetching data for resource_id: 0419a647-55f3-41a5-b8d4-ee3c50553aca
Fetching data for resource_id: 65ba896b-e3f3-437

,調査年月日,区分,流入依存度,社会動態_増減
2,2025-12-31,日本人_東区,0.672103,164
3,2025-12-31,日本人_博多区,0.667881,-120
10,2025-12-31,日本人_うち西部出張所,0.575758,11
4,2025-12-31,日本人_中央区,0.558257,-124
5,2025-12-31,日本人_南区,0.533333,103
...,...,...,...,...
488,2024-01-31,日本人_南区,0.612903,49
492,2024-01-31,日本人_西区,0.579528,14
490,2024-01-31,日本人_早良区,0.557743,119
489,2024-01-31,日本人_城南区,0.528226,7
